# 08 — Ablations (A1, A2, A3, A5, A6)
All under the pre-registered protocol (notebook 02). Primary setting:
**FD004, regime-norm, no buffer**. A1 runs over the 5 pre-registered seeds;
the other sweeps use seed 42 to keep compute manageable.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.loaders import prepare, load, load_rul, SENSORS, SEEDS, HEALTHY_RUL
from src.scores import GeometricModel
from src import metrics

## A1 — Regime normalization (the main ablation), 5 seeds

In [2]:
rows = []
for seed in SEEDS:
    for regime in [False, True]:
        d = prepare('FD004', seed=seed, by_regime=regime)
        gm = GeometricModel().fit(d['fit'])
        sc = gm.score(d['val'])
        rows.append(dict(seed=seed, norm='per-regime' if regime else 'global',
                         pos=metrics.auroc(sc['pos'], d['val'].rul, use_buffer=False),
                         net=metrics.auroc(sc['net'], d['val'].rul, use_buffer=False)))
a1 = pd.DataFrame(rows).groupby('norm')[['pos', 'net']].agg(['mean', 'std']).round(3)
a1.to_csv('../results/tables/08_A1_regime.csv'); a1

pos           net       
             mean    std   mean    std
norm                                  
global      0.587  0.013  0.514  0.007
per-regime  0.977  0.002  0.962  0.003

## A2 — Smoothing window w (velocity), seed 42

In [3]:
d = prepare('FD001', seed=42)
res = {}
for w in [5, 10, 15, 30]:
    gm = GeometricModel(w=w).fit(d['fit'])
    sc = gm.score(d['val'])
    res[w] = metrics.auroc(sc['vel'], d['val'].rul, use_buffer=False)
a2 = pd.Series(res, name='velocity AUROC (FD001, no buffer)').round(3)
a2.to_csv('../results/tables/08_A2_window.csv'); print(a2)

5     0.659
10    0.799
15    0.871
30    0.933
Name: velocity AUROC (FD001, no buffer), dtype: float64


## A3 — Score-set ablation (what each score adds), seed 42, FD004

In [4]:
from sklearn.ensemble import IsolationForest
d = prepare('FD004', seed=42)
gm = GeometricModel().fit(d['fit'])
sc_f, sc_v = gm.score(d['fit']), gm.score(d['val'])
hm = (d['fit'].rul > HEALTHY_RUL).values
out = {}
for name, F in [('pos', ['pos']), ('pos+vel', ['pos', 'vel']), ('pos+vel+net', ['pos', 'vel', 'net'])]:
    if len(F) == 1:
        s = sc_v[F[0]]
    else:
        iso = IsolationForest(n_estimators=200, random_state=0).fit(sc_f.loc[hm, F])
        s = -iso.score_samples(sc_v[F])
    out[name] = metrics.auroc(s, d['val'].rul, use_buffer=False)
a3 = pd.Series(out, name='AUROC FD004 no-buffer').round(3)
a3.to_csv('../results/tables/08_A3_scoreset.csv'); print(a3)

pos            0.979
pos+vel        0.973
pos+vel+net    0.975
Name: AUROC FD004 no-buffer, dtype: float64


## A5 — Cost-based re-ranking (c_FN/c_FP ∈ {10, 50, 100}), seed 42

In [5]:
d = prepare('FD004', seed=42)
gm = GeometricModel().fit(d['fit'])
sc_v = gm.score(d['val'])
cand = {'position': sc_v['pos'].values, 'net-displacement': sc_v['net'].values,
        'z-s11': d['val'].s11.abs().values}
rows = {}
for name, s in cand.items():
    thr = np.quantile(s[(d['val'].rul > HEALTHY_RUL).values], 0.99)  # 1% healthy FP budget
    rows[name] = {f'c_FN/c_FP={r}': metrics.cost_weighted(s, d['val'].rul, thr, c_fn=r, use_buffer=False)[0]
                  for r in [10, 50, 100]}
a5 = pd.DataFrame(rows).T
a5.to_csv('../results/tables/08_A5_cost.csv'); a5

,c_FN/c_FP=10,c_FN/c_FP=50,c_FN/c_FP=100
position,1684,3764,6364
net-displacement,2357,3997,6047
z-s11,2674,10154,19504


## A6 — θ and protocol sensitivity, seed 42

In [6]:
rows = {}
for theta in [20, 30, 50]:
    for buf in [True, False]:
        rows[(theta, 'buffer' if buf else 'no-buffer')] = round(
            metrics.auroc(sc_v['net'], d['val'].rul, theta=theta, use_buffer=buf), 3)
a6 = pd.Series(rows, name='net-displacement AUROC (FD004)')
a6.to_csv('../results/tables/08_A6_theta.csv'); print(a6)

20  buffer       0.999
    no-buffer    0.961
30  buffer       0.998
    no-buffer    0.963
50  buffer       0.991
    no-buffer    0.962
Name: net-displacement AUROC (FD004), dtype: float64
